<a href="https://colab.research.google.com/github/gilbertoag2007/fiap-tech-challenge-fase3/blob/main/tech_challenge_fase_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#AGRUPANDO AS 5 PARTES DO DATASET

In [20]:
# Clone do repositório no GITHUB
!git clone https://github.com/gilbertoag2007/fiap-tech-challenge-fase3.git

fatal: destination path 'fiap-tech-challenge-fase3' already exists and is not an empty directory.


In [21]:
# ============================================================
# 1. IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

import pandas as pd
import re
import spacy
from pathlib import Path

!pip install -q pandas pyarrow openpyxl
!pip install pandas openpyxl spacy
!python -m spacy download pt_core_news_lg



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 757.9 kB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# CARREGAMENTO DOS DADOS

In [22]:
"""
===========================================================================
OBJETIVO
===========================================================================

Este script reúne vários arquivos XLSX que foram divididos devido ao
limite de tamanho do GitHub (ex.: arquivos maiores que 80 MB).

Ao final será gerado um único dataset que servirá de entrada para as
etapas de:

1. Identificação de dados pessoais (PHI)
2. Anonimização
3. Limpeza dos dados
4. Fine-Tuning de uma LLM

Bibliotecas necessárias:

pip install pandas openpyxl

===========================================================================
"""

# ==========================================================================
# CONFIGURAÇÕES
# ==========================================================================

# Pasta onde estão os arquivos divididos
PASTA_DADOS = "/content/fiap-tech-challenge-fase3/data"

# Nome do arquivo de saída
ARQUIVO_FINAL = "dataset_medico_completo.xlsx"

# ==========================================================================
# Localiza todos os arquivos XLSX da pasta
# ==========================================================================

print("=" * 70)
print("LOCALIZANDO ARQUIVOS...")
print("=" * 70)

# Procura todos os arquivos .xlsx
arquivos = sorted(Path(PASTA_DADOS).glob("*.xlsx"))

# Verifica se encontrou arquivos
if len(arquivos) == 0:
    raise Exception("Nenhum arquivo XLSX encontrado.")

print(f"Foram encontrados {len(arquivos)} arquivos.\n")

# ==========================================================================
# Leitura dos arquivos
# ==========================================================================

print("=" * 70)
print("LENDO ARQUIVOS...")
print("=" * 70)

lista_dataframes = []

for arquivo in arquivos:

    print(f"Lendo {arquivo.name}")

    # Carrega o arquivo para um DataFrame
    df = pd.read_excel(
        arquivo,
        engine="openpyxl"
    )

    # Guarda o DataFrame em memória
    lista_dataframes.append(df)

# ==========================================================================
# Junta todos os DataFrames
# ==========================================================================

print("\nUnindo arquivos...")

dataframe_original = pd.concat(
    lista_dataframes,
    ignore_index=True
)

print("Arquivos unidos com sucesso!")



LOCALIZANDO ARQUIVOS...
Foram encontrados 4 arquivos.

LENDO ARQUIVOS...
Lendo dataset_medico_part001.xlsx
Lendo dataset_medico_part002.xlsx
Lendo dataset_medico_part003.xlsx
Lendo dataset_medico_part004.xlsx

Unindo arquivos...
Arquivos unidos com sucesso!


In [23]:
# ==========================================================================
# Limpeza básica
# ==========================================================================

print("\nRealizando limpeza inicial...")

# Remove linhas totalmente vazias
dataframe_original.dropna(
    how="all",
    inplace=True
)

# Remove registros duplicados
dataframe_original.drop_duplicates(
    inplace=True
)

# Reinicia a numeração do índice
dataframe_original.reset_index(
    drop=True,
    inplace=True
)



Realizando limpeza inicial...


In [24]:
# ==========================================================================
# REDUZ O TAMANHO DO DATASET PARA AGILIZAR OS TESTES EM DESENVOLVIMENTO
# ==========================================================================

dataframe_reduzido = dataframe_original.sample(frac=0.01, random_state=42)


In [25]:
# ==========================================================================
# Informações do dataset
# ==========================================================================

print("\nResumo do dataset")

print("-" * 60)

print(f"Quantidade de registros : {len(dataframe_reduzido):,}")

print(f"Quantidade de colunas   : {len(dataframe_reduzido.columns)}")

print("\nColunas encontradas:\n")

for coluna in dataframe_reduzido.columns:
    print(f" - {coluna}")

# ==========================================================================
# Verificação de valores nulos
# ==========================================================================

print("\nValores ausentes por coluna\n")

print(dataframe_reduzido.isnull().sum())

# ==========================================================================
# Estatísticas básicas
# ==========================================================================

print("\nPrimeiros registros:\n")

print(dataframe_reduzido.head())


Resumo do dataset
------------------------------------------------------------
Quantidade de registros : 3,841
Quantidade de colunas   : 6

Colunas encontradas:

 - id
 - pergunta_com_dados_pessoais
 - resposta_formatada
 - condicao
 - especialidade_medica
 - tipo_pergunta

Valores ausentes por coluna

id                             0
pergunta_com_dados_pessoais    0
resposta_formatada             0
condicao                       0
especialidade_medica           0
tipo_pergunta                  0
dtype: int64

Primeiros registros:

            id                        pergunta_com_dados_pessoais  \
56799   331651             Demência. O que pode levar à demência?   
299108  429542  Meu namorado disse que há um tempo atrás foi d...   
172048  338711  Estou em um relacionamento relativamente recen...   
21162    27410  O que pode causar zumbido pulsátil , sendo que...   
82076   117978  exercício físico diminui a glicemia e a glicos...   

                                       respost

In [26]:

# ==========================================================================
# Salva o dataset consolidado
# ==========================================================================

print("\nSalvando arquivo consolidado...")

dataframe_reduzido.to_excel(
    ARQUIVO_FINAL,
    index=False,
    engine="openpyxl"
)

print("\nArquivo salvo com sucesso!")

print(f"\nArquivo gerado: {ARQUIVO_FINAL}")



Salvando arquivo consolidado...

Arquivo salvo com sucesso!

Arquivo gerado: dataset_medico_completo.xlsx


# DECTECTAR PHI - Protected Health Information

In [27]:


import re
import pandas as pd
import spacy
# ============================================================
# 1. CARREGAR MODELO NLP
# ============================================================

nlp = spacy.load("pt_core_news_lg")


# ============================================================
# 2. LISTA DE TERMOS QUE NÃO DEVEM SER CONSIDERADOS
#    NOMES DE PESSOAS
# ============================================================

TERMOS_MEDICOS = {
    "esclerose",
    "esquizofrenia",
    "bipolar",
    "bipolaridade",
    "artrose",
    "discopatias",
    "discopatia",
    "diagnóstico",
    "diagnostico",
    "paciente",
    "pacientes",
    "médico",
    "medico",
    "médica",
    "medica",
    "doença",
    "doenca",
    "doenças",
    "doencas",
    "neurônio",
    "neuronio",
    "cirurgia",
    "cirúrgico",
    "cirurgico",
    "consulta",
    "teleconsulta",
    "tratamento",
    "procedimento",
    "procedimentos",
    "hospital",
    "clínica",
    "clinica",
    "artrose",
    "câncer",
    "cancer",
    "diabetes",
    "hipertensão",
    "hipertensao",
    "depressão",
    "depressao",
    "ansiedade",
    "ela"
}


# ============================================================
# 3. PADRÕES REGEX
# ============================================================

PADROES = {

    # --------------------------------------------------------
    # CPF
    # --------------------------------------------------------

    "CPF": re.compile(
        r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b"
    ),


    # --------------------------------------------------------
    # E-MAIL
    # --------------------------------------------------------

    "EMAIL": re.compile(
        r"\b[A-Za-z0-9._%+-]+@"
        r"[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"
    ),


    # --------------------------------------------------------
    # TELEFONE
    # --------------------------------------------------------

    "TELEFONE": re.compile(
        r"(?<!\d)"
        r"(?:\+55\s?)?"
        r"(?:\(?\d{2}\)?\s?)?"
        r"(?:9\d{4}|\d{4})[-\s]?\d{4}"
        r"(?!\d)"
    ),


    # --------------------------------------------------------
    # CEP
    # --------------------------------------------------------

    "CEP": re.compile(
        r"\b\d{5}-?\d{3}\b"
    ),


    # --------------------------------------------------------
    # DATA
    # --------------------------------------------------------

    "DATA": re.compile(
        r"\b(?:0?[1-9]|[12]\d|3[01])"
        r"[/\-]"
        r"(?:0?[1-9]|1[0-2])"
        r"[/\-]"
        r"(?:19|20)\d{2}\b"
    ),


    # --------------------------------------------------------
    # RG
    # --------------------------------------------------------

    "RG": re.compile(
        r"\b\d{1,2}\.?\d{3}\.?\d{3}-?[0-9Xx]\b"
    )
}


# ============================================================
# 4. VALIDAÇÃO DE CPF
# ============================================================

def cpf_valido(cpf):

    # Remove caracteres não numéricos
    numeros = re.sub(r"\D", "", cpf)

    # CPF precisa ter 11 dígitos
    if len(numeros) != 11:
        return False

    # Elimina CPFs como:
    # 11111111111
    # 22222222222
    # etc.

    if numeros == numeros[0] * 11:
        return False


    # --------------------------------------------
    # Primeiro dígito verificador
    # --------------------------------------------

    soma = sum(
        int(numeros[i]) * (10 - i)
        for i in range(9)
    )

    resto = soma % 11

    digito1 = 0 if resto < 2 else 11 - resto


    if int(numeros[9]) != digito1:
        return False


    # --------------------------------------------
    # Segundo dígito verificador
    # --------------------------------------------

    soma = sum(
        int(numeros[i]) * (11 - i)
        for i in range(10)
    )

    resto = soma % 11

    digito2 = 0 if resto < 2 else 11 - resto


    if int(numeros[10]) != digito2:
        return False


    return True


# ============================================================
# 5. DETECTAR REGEX
# ============================================================

def detectar_regex(texto):

    encontrados = []


    for tipo, padrao in PADROES.items():

        ocorrencias = padrao.findall(texto)


        # --------------------------------------------
        # CPF
        # --------------------------------------------

        if tipo == "CPF":

            for ocorrencia in ocorrencias:

                if cpf_valido(ocorrencia):

                    encontrados.append(tipo)

                    break


        # --------------------------------------------
        # Outros padrões
        # --------------------------------------------

        elif ocorrencias:

            encontrados.append(tipo)


    return encontrados


# ============================================================
# 6. FUNÇÃO PARA VERIFICAR SE UM TERMO É MÉDICO
# ============================================================

def eh_termo_medico(texto):

    texto = texto.lower().strip()

    return texto in TERMOS_MEDICOS


# ============================================================
# 7. DETECTAR NOME DE PESSOA
# ============================================================

def detectar_pessoa(entidade, texto):

    valor = entidade.text.strip()

    valor_lower = valor.lower()


    # --------------------------------------------------------
    # Não considerar termos médicos como nomes
    # --------------------------------------------------------

    if valor_lower in TERMOS_MEDICOS:

        return False


    # --------------------------------------------------------
    # Nome deve possuir pelo menos duas palavras
    #
    # Exemplo:
    #
    # Leonardo Castro Duarte Alves
    #
    # --------------------------------------------------------

    palavras = valor.split()

    if len(palavras) < 2:

        return False


    # --------------------------------------------------------
    # Verificar se possui palavras com inicial maiúscula
    # --------------------------------------------------------

    palavras_maiusculas = sum(
        1
        for palavra in palavras
        if palavra[:1].isupper()
    )


    if palavras_maiusculas < 2:

        return False


    return True


# ============================================================
# 8. DETECTAR LOCALIZAÇÃO
# ============================================================

def detectar_localizacao(entidade, texto):

    valor = entidade.text.strip()

    valor_lower = valor.lower()


    # --------------------------------------------------------
    # Evitar termos médicos
    # --------------------------------------------------------

    if valor_lower in TERMOS_MEDICOS:

        return False


    # --------------------------------------------------------
    # Contextos que aumentam a confiança
    # --------------------------------------------------------

    padroes_contexto = [

        r"\bresidente em\b",

        r"\bmora em\b",

        r"\bmorador de\b",

        r"\bnatural de\b",

        r"\bnascido em\b",

        r"\bnascida em\b",

        r"\bproveniente de\b",

        r"\bprocedente de\b",

        r"\bendereço\b",

        r"\bdomicílio\b",

        r"\bdomicilio\b",

        r"\bcidade de\b",

        r"\bestado de\b",

        r"\bmunicípio de\b",

        r"\bmunicipio de\b",

        r"\bresidência em\b",

        r"\bresidencia em\b"
    ]


    for padrao in padroes_contexto:

        if re.search(
            padrao,
            texto,
            re.IGNORECASE
        ):

            return True


    # --------------------------------------------------------
    # Sem contexto explícito, não classificar como PII
    # --------------------------------------------------------

    return False


# ============================================================
# 9. DETECTAR ORGANIZAÇÃO
# ============================================================

def detectar_organizacao(entidade, texto):

    valor = entidade.text.strip()

    valor_lower = valor.lower()


    # --------------------------------------------------------
    # Evitar falsos positivos relacionados à área médica
    # --------------------------------------------------------

    if valor_lower in TERMOS_MEDICOS:

        return False


    # --------------------------------------------------------
    # Palavras que normalmente indicam organização
    # --------------------------------------------------------

    indicadores = [

        "hospital",

        "clínica",

        "clinica",

        "universidade",

        "instituto",

        "laboratório",

        "laboratorio",

        "empresa",

        "fundação",

        "fundacao",

        "associação",

        "associacao",

        "faculdade",

        "escola",

        "prefeitura",

        "secretaria",

        "ministério",

        "ministerio"
    ]


    # --------------------------------------------------------
    # Verifica se o próprio nome possui indicador
    # --------------------------------------------------------

    if any(
        indicador in valor_lower
        for indicador in indicadores
    ):

        return True


    # --------------------------------------------------------
    # Verifica contexto
    # --------------------------------------------------------

    padroes_contexto = [

        r"\bno hospital\b",

        r"\bna clínica\b",

        r"\bna clinica\b",

        r"\bno laboratório\b",

        r"\bno laboratorio\b",

        r"\bda universidade\b",

        r"\bdo instituto\b",

        r"\bna empresa\b",

        r"\bno instituto\b"
    ]


    for padrao in padroes_contexto:

        if re.search(
            padrao,
            texto,
            re.IGNORECASE
        ):

            return True


    return False


# ============================================================
# 10. DETECTAR ENTIDADES COM CONTEXTO
# ============================================================

def detectar_entidades(texto):

    entidades_detectadas = []

    doc = nlp(texto)


    for entidade in doc.ents:

        # ================================================
        # PESSOA
        # ================================================

        if entidade.label_ == "PER":

            if detectar_pessoa(
                entidade,
                texto
            ):

                entidades_detectadas.append(
                    "NOME_PESSOA"
                )


        # ================================================
        # ORGANIZAÇÃO
        # ================================================

        elif entidade.label_ == "ORG":

            if detectar_organizacao(
                entidade,
                texto
            ):

                entidades_detectadas.append(
                    "ORGANIZACAO"
                )


        # ================================================
        # LOCALIZAÇÃO
        # ================================================

        elif entidade.label_ in [
            "LOC",
            "GPE"
        ]:

            if detectar_localizacao(
                entidade,
                texto
            ):

                entidades_detectadas.append(
                    "LOCALIZACAO"
                )


    return list(
        set(entidades_detectadas)
    )


# ============================================================
# 11. ANALISAR TEXTO
# ============================================================

def analisar_texto(valor):

    # --------------------------------------------
    # Valor vazio
    # --------------------------------------------

    if pd.isna(valor):

        return []


    texto = str(valor).strip()


    if not texto:

        return []


    tipos = []


    # --------------------------------------------
    # Regex
    # --------------------------------------------

    tipos.extend(
        detectar_regex(texto)
    )


    # --------------------------------------------
    # NER + regras contextuais
    # --------------------------------------------

    tipos.extend(
        detectar_entidades(texto)
    )


    # --------------------------------------------
    # Remover duplicidades
    # --------------------------------------------

    return sorted(
        list(set(tipos))
    )

In [28]:
# ============================================================
# ANÁLISE DE DADOS PESSOAIS / PII / PHI
# ============================================================
#
# Objetivo:
#
# 1. Percorrer as colunas definidas em "colunas_analisar"
# 2. Analisar cada registro
# 3. Identificar possíveis dados pessoais
# 4. Classificar as detecções por nível de confiança
# 5. Contar os registros identificados por tipo
# 6. Calcular percentuais
# 7. Mostrar o resultado diretamente no notebook
#
#
# NÍVEIS DE CONFIANÇA
#
# ALTA:
#   CPF
#   EMAIL
#   TELEFONE
#   CEP
#   DATA
#   RG
#
# MÉDIA:
#   NOME_PESSOA
#   LOCALIZACAO
#   ORGANIZACAO
#
# IMPORTANTE:
#
# Um mesmo registro pode possuir vários tipos de PII.
#
# Exemplo:
#
# "Leonardo Castro Duarte Alves,
#  CPF 60763507624,
#  nascido em 21/03/1976"
#
# Será contabilizado:
#
# NOME_PESSOA -> 1
# CPF         -> 1
# DATA        -> 1
#
# Porém o registro será contado somente uma vez em:
#
# registros_com_dados_pessoais
#
# ============================================================


# ============================================================
# 1. DEFINIR AS COLUNAS QUE SERÃO ANALISADAS
# ============================================================

colunas_analisar = [
    "pergunta_com_dados_pessoais",
    "resposta_formatada"
]


# ============================================================
# 2. CONFIGURAÇÃO DOS NÍVEIS DE CONFIANÇA
# ============================================================

# Tipos que consideramos de alta confiança porque são
# identificados por padrões determinísticos / regex.

TIPOS_ALTA_CONFIANCA = {
    "CPF",
    "EMAIL",
    "TELEFONE",
    "CEP",
    "DATA",
    "RG"
}


# Tipos identificados normalmente por NER e regras
# contextuais.

TIPOS_MEDIA_CONFIANCA = {
    "NOME_PESSOA",
    "LOCALIZACAO",
    "ORGANIZACAO"
}


# ============================================================
# 3. FUNÇÃO PARA DETERMINAR A CONFIANÇA
# ============================================================

def determinar_confianca(tipo):

    if tipo in TIPOS_ALTA_CONFIANCA:

        return "ALTA"

    elif tipo in TIPOS_MEDIA_CONFIANCA:

        return "MEDIA"

    else:

        return "BAIXA"


# ============================================================
# 4. LISTA PARA ARMAZENAR OS RESULTADOS
# ============================================================

resultados = []


# ============================================================
# 5. PERCORRER AS COLUNAS
# ============================================================

for coluna in colunas_analisar:

    print("\n")
    print("=" * 100)
    print(f"ANALISANDO COLUNA: {coluna}")
    print("=" * 100)


    # --------------------------------------------------------
    # Verificar se a coluna realmente existe
    # --------------------------------------------------------

    if coluna not in dataframe_reduzido.columns:

        print(
            f"ATENÇÃO: a coluna '{coluna}' "
            f"não existe no DataFrame."
        )

        continue


    # --------------------------------------------------------
    # Total de registros
    # --------------------------------------------------------

    total_registros = len(dataframe_reduzido)


    # --------------------------------------------------------
    # Número de registros que possuem pelo menos
    # um dado pessoal
    # --------------------------------------------------------

    registros_com_dados = 0


    # --------------------------------------------------------
    # Número de registros classificados como ALTA confiança
    # --------------------------------------------------------

    registros_alta_confianca = 0


    # --------------------------------------------------------
    # Número de registros classificados como MÉDIA confiança
    # --------------------------------------------------------

    registros_media_confianca = 0


    # --------------------------------------------------------
    # Dicionários para contagem
    # --------------------------------------------------------

    contadores = {}

    contadores_confianca = {
        "ALTA": 0,
        "MEDIA": 0,
        "BAIXA": 0
    }


    # --------------------------------------------------------
    # Exemplos seguros para auditoria
    #
    # NÃO armazenaremos o conteúdo da pergunta/resposta.
    # Apenas linha, tipo e confiança.
    # --------------------------------------------------------

    exemplos = []


    # ========================================================
    # 6. PERCORRER OS REGISTROS
    # ========================================================

    for indice, valor in dataframe_reduzido[coluna].items():


        # ----------------------------------------------------
        # Executar o detector
        # ----------------------------------------------------

        tipos = analisar_texto(valor)


        # ----------------------------------------------------
        # Se encontrou algum tipo de dado pessoal
        # ----------------------------------------------------

        if tipos:

            # Conta o registro somente uma vez
            registros_com_dados += 1


            # -----------------------------------------------
            # Identificar a maior confiança daquele registro
            # -----------------------------------------------

            confiancas_registro = []


            # ===============================================
            # 7. CONTAR CADA TIPO
            # ===============================================

            for tipo in tipos:

                # -------------------------------------------
                # Inicializar contador
                # -------------------------------------------

                if tipo not in contadores:

                    contadores[tipo] = 0


                # -------------------------------------------
                # Incrementar contador
                # -------------------------------------------

                contadores[tipo] += 1


                # -------------------------------------------
                # Determinar confiança
                # -------------------------------------------

                confianca = determinar_confianca(tipo)

                confiancas_registro.append(confianca)


                # -------------------------------------------
                # Contabilizar confiança
                # -------------------------------------------

                contadores_confianca[confianca] += 1


            # =================================================
            # 8. DETERMINAR A MAIOR CONFIANÇA DO REGISTRO
            # =================================================

            if "ALTA" in confiancas_registro:

                maior_confianca = "ALTA"

            elif "MEDIA" in confiancas_registro:

                maior_confianca = "MEDIA"

            else:

                maior_confianca = "BAIXA"


            # ------------------------------------------------
            # Contabilizar registros por confiança
            # ------------------------------------------------

            if maior_confianca == "ALTA":

                registros_alta_confianca += 1

            elif maior_confianca == "MEDIA":

                registros_media_confianca += 1


            # =================================================
            # 9. GUARDAR ATÉ 5 EXEMPLOS
            # =================================================
            #
            # IMPORTANTE:
            #
            # Não armazenamos o conteúdo do texto.
            #
            # Isso evita que dados pessoais/médicos sejam
            # impressos ou armazenados acidentalmente.
            # =================================================

            if len(exemplos) < 5:

                exemplos.append({

                    "linha": indice + 2,

                    "tipos": ", ".join(tipos),

                    "confianca": maior_confianca

                })


    # ========================================================
    # 10. CALCULAR PERCENTUAL DE REGISTROS SUSPEITOS
    # ========================================================

    if total_registros > 0:

        percentual_suspeito = (
            registros_com_dados /
            total_registros
        ) * 100

    else:

        percentual_suspeito = 0


    # ========================================================
    # 11. CALCULAR PERCENTUAIS POR CONFIANÇA
    # ========================================================

    if total_registros > 0:

        percentual_alta = (
            registros_alta_confianca /
            total_registros
        ) * 100


        percentual_media = (
            registros_media_confianca /
            total_registros
        ) * 100

    else:

        percentual_alta = 0
        percentual_media = 0


    # ========================================================
    # 12. EXIBIR RESUMO DA COLUNA
    # ========================================================

    print(
        f"\nTotal de registros: "
        f"{total_registros:,}"
    )


    print(
        f"Registros com dados pessoais: "
        f"{registros_com_dados:,}"
    )


    print(
        f"Percentual suspeito: "
        f"{percentual_suspeito:.2f}%"
    )


    # ========================================================
    # 13. RESUMO POR NÍVEL DE CONFIANÇA
    # ========================================================

    print("\n")
    print("CLASSIFICAÇÃO POR CONFIANÇA")
    print("-" * 100)


    print(
        f"ALTA   : "
        f"{registros_alta_confianca:>10,} "
        f"({percentual_alta:.2f}%)"
    )


    print(
        f"MEDIA  : "
        f"{registros_media_confianca:>10,} "
        f"({percentual_media:.2f}%)"
    )


    # ========================================================
    # 14. RESULTADO POR TIPO DE DADO
    # ========================================================

    print("\n")
    print("DADOS PESSOAIS IDENTIFICADOS")
    print("-" * 100)


    if contadores:

        print(
            f"{'TIPO':<25}"
            f"{'CONFIANÇA':<15}"
            f"{'QUANTIDADE':>15}"
            f"{'PERCENTUAL':>18}"
        )

        print("-" * 100)


        # ----------------------------------------------------
        # Ordenar tipos alfabeticamente
        # ----------------------------------------------------

        for tipo, quantidade in sorted(
            contadores.items()
        ):


            # -----------------------------------------------
            # Determinar confiança
            # -----------------------------------------------

            confianca = determinar_confianca(tipo)


            # -----------------------------------------------
            # Calcular percentual
            # -----------------------------------------------

            if total_registros > 0:

                percentual_tipo = (
                    quantidade /
                    total_registros
                ) * 100

            else:

                percentual_tipo = 0


            # -----------------------------------------------
            # Imprimir
            # -----------------------------------------------

            print(
                f"{tipo:<25}"
                f"{confianca:<15}"
                f"{quantidade:>15,}"
                f"{percentual_tipo:>17.2f}%"
            )


    else:

        print(
            "Nenhum dado pessoal identificado."
        )


    # ========================================================
    # 15. MOSTRAR EXEMPLOS DE DETECÇÕES
    # ========================================================
    #
    # Não mostramos o conteúdo original.
    #
    # Exemplo:
    #
    # Linha 125
    # Tipos: CPF, NOME_PESSOA
    # Confiança: ALTA
    #
    # ========================================================

    if exemplos:

        print("\n")
        print("EXEMPLOS DE DETECÇÕES")
        print("-" * 100)

        for exemplo in exemplos:

            print(
                f"Linha: {exemplo['linha']}"
            )

            print(
                f"Tipos: {exemplo['tipos']}"
            )

            print(
                f"Confiança: {exemplo['confianca']}"
            )

            print("-" * 100)


    # ========================================================
    # 16. ARMAZENAR RESULTADO DA COLUNA
    # ========================================================

    resultados.append({

        "coluna": coluna,

        "total_registros":
            total_registros,

        "registros_com_dados_pessoais":
            registros_com_dados,

        "percentual_suspeito":
            round(
                percentual_suspeito,
                2
            ),

        "registros_alta_confianca":
            registros_alta_confianca,

        "registros_media_confianca":
            registros_media_confianca,

        "quantidades_por_tipo":
            contadores,

        "quantidades_por_confianca":
            contadores_confianca,

        "exemplos":
            exemplos

    })


# ============================================================
# 17. RESUMO FINAL DE TODAS AS COLUNAS
# ============================================================

print("\n\n")

print("=" * 100)
print("RESUMO FINAL DA ANÁLISE")
print("=" * 100)


for resultado in resultados:

    print("\n")

    print(
        f"COLUNA: {resultado['coluna']}"
    )


    print(
        f"Total de registros: "
        f"{resultado['total_registros']:,}"
    )


    print(
        f"Registros com dados pessoais: "
        f"{resultado['registros_com_dados_pessoais']:,}"
    )


    print(
        f"Percentual suspeito: "
        f"{resultado['percentual_suspeito']:.2f}%"
    )


    print("\nTipos identificados:")


    for tipo, quantidade in sorted(
        resultado["quantidades_por_tipo"].items()
    ):


        if resultado["total_registros"] > 0:

            percentual = (
                quantidade /
                resultado["total_registros"]
            ) * 100

        else:

            percentual = 0


        confianca = determinar_confianca(tipo)


        print(
            f"  {tipo:<25}"
            f"{confianca:<10}"
            f"{quantidade:>10,} "
            f"({percentual:.2f}%)"
        )



ANALISANDO COLUNA: pergunta_com_dados_pessoais

Total de registros: 3,841
Registros com dados pessoais: 245
Percentual suspeito: 6.38%


CLASSIFICAÇÃO POR CONFIANÇA
----------------------------------------------------------------------------------------------------
ALTA   :        177 (4.61%)
MEDIA  :         68 (1.77%)


DADOS PESSOAIS IDENTIFICADOS
----------------------------------------------------------------------------------------------------
TIPO                     CONFIANÇA           QUANTIDADE        PERCENTUAL
----------------------------------------------------------------------------------------------------
CEP                      ALTA                         4             0.10%
CPF                      ALTA                         1             0.03%
DATA                     ALTA                       172             4.48%
LOCALIZACAO              MEDIA                       24             0.62%
NOME_PESSOA              MEDIA                      238             6.20%